# Model Selection Results

Viewer for the latest pipeline run. Produce a run first, from the `research` folder:

```
..\.venv\Scripts\python -m modeling.build_features            # Sheets -> data/features.parquet
..\.venv\Scripts\python -m modeling.run_pipeline --quick      # smoke test
..\.venv\Scripts\python -m modeling.run_pipeline --device auto # full run
```

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from modeling.inference import latest_run_dir, list_models, load_model, e1rm_to_weight

run_dir = latest_run_dir(include_quick=True)   # or Path('artifacts/<run_id>')
print(run_dir)
list_models(run_dir)

## Reports

In [ ]:
display(Markdown((run_dir / 'reports' / 'model_selection.md').read_text(encoding='utf-8')))

In [ ]:
display(Markdown((run_dir / 'reports' / 'feature_selection.md').read_text(encoding='utf-8')))

## Holdout predictions for one exercise

In [ ]:
hold = pd.read_parquet(run_dir / 'predictions' / 'holdout.parquet')
summary = json.loads((run_dir / 'summary.json').read_text())
best = [summary.get('production_model'), summary['best_model']]

exercise, set_number = hold['exercise'].value_counts().index[0], 1   # change to any lift
h = hold[(hold.exercise == exercise) & (hold.set_number == set_number)].sort_values('date')
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(h.date, h.y, 'o-', label='actual est_1rm', color='black')
for m in dict.fromkeys(m for m in ['naive_last', *best] if m):
    ax.plot(h.date, h[m], '.--', label=m)
ax.set_title(f'{exercise} set {set_number} (holdout)')
ax.legend()
plt.show()

## Convert a predicted est_1rm into a working weight

Uses the reps x RPE ratio table derived from your own sets (`ratio_table.csv`).

In [ ]:
e1rm = 250.0
pd.DataFrame({reps: [e1rm_to_weight(e1rm, reps, rpe, run_dir=run_dir, round_to=2.5) for rpe in (8, 9, 10)]
              for reps in (5, 6, 8, 10, 12)}, index=['RPE 8', 'RPE 9', 'RPE 10'])

## Predict your next workout

History defaults to the sheet snapshot saved by `build_features` (`data/raw_sets.parquet`). Pass `history=load_raw_data()` to use the live sheet.

In [ ]:
from modeling.inference import make_future_rows, predict_next

future = make_future_rows('2026-11-24', 'Push', [
    {'exercise': 'Bench Press', 'sets': 2, 'type': 'Hypertrophy'},
    {'exercise': 'Lat Pulldown', 'sets': 3, 'type': 'Hypertrophy'},
])
preds = predict_next(future, run_dir=run_dir)
preds['weight_8_reps_rpe10'] = e1rm_to_weight(preds['predicted_est_1rm'], reps=8, rpe=10, run_dir=run_dir, round_to=2.5)
preds